# Customer Analytics using Azure Databricks

## 1. Load Dataset

In [0]:
df = spark.read.csv(
    "/Volumes/customeranalyticsdb/default/raw-data/online_retail.csv",
    header=True,
    inferSchema=True
)

In [0]:
df.show(5)

In [0]:
df.printSchema()

In [0]:
print("Rows:", df.count())
print("Columns:", len(df.columns))

## 2. Data Exploration

In [0]:
from pyspark.sql.functions import col
print("Missing Customer IDs:",df.filter(col("CustomerID").isNull()).count())
print("Negative Quantities:",df.filter(col("Quantity") < 0).count())
print("Invalid Prices:",df.filter(col("UnitPrice") <= 0).count())
print("Total Rows:", df.count())
print("Unique Rows:", df.dropDuplicates().count())

## 3. Data Cleaning

In [0]:
from pyspark.sql.functions import col
clean_df = (

    df
    .filter(col("CustomerID").isNotNull())
    .filter(col("Quantity") > 0)
    .filter(col("UnitPrice") > 0)
    .dropDuplicates()
    .withColumn("CustomerID", col("CustomerID").cast("int"))
)

In [0]:
print("Original Rows:", df.count())
print("Clean Rows:", clean_df.count())

In [0]:
print("Total Rows:", df.count())
print("Unique Rows:", df.dropDuplicates().count())

In [0]:
print("Missing CustomerID:",
      clean_df.filter(col("CustomerID").isNull()).count())

print("Negative Quantity:",
      clean_df.filter(col("Quantity") <= 0).count())

print("Invalid Price:",
      clean_df.filter(col("UnitPrice") <= 0).count())

## 4. Revenue Transformation

In [0]:
from pyspark.sql.functions import col

clean_df = clean_df.withColumn(
    "Revenue",
    col("Quantity") * col("UnitPrice")
)

In [0]:
clean_df.select(
    "Quantity",
    "UnitPrice",
    "Revenue"
).show(10)

In [0]:
clean_df.printSchema()

In [0]:
from pyspark.sql.functions import sum, round, col

clean_df.select(
    round(sum("Revenue"), 0).cast("long").alias("Total Revenue")
).show()

In [0]:
clean_df.select("CustomerID").distinct().count()

## 5. Top Customers Analysis

In [0]:
from pyspark.sql.functions import round

top_customers = (
    clean_df
    .groupBy("CustomerID")
    .agg(
        round(sum("Revenue"), 0).cast("long").alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

top_customers.show(10)

## 6. Revenue by Country Analysis

In [0]:
country_revenue = (
    clean_df
    .groupBy("Country")
    .agg(
        round(sum("Revenue"), 0).cast("long").alias("TotalRevenue")
    )
    .orderBy(col("TotalRevenue").desc())
)

country_revenue.show(10)

In [0]:
clean_df.select("Country").distinct().count()

In [0]:
from pyspark.sql.functions import year, month

sales_trend = (
    clean_df
    .withColumn("Year", year("InvoiceDate"))
    .withColumn("Month", month("InvoiceDate"))
)

## 7. Monthly Sales Analysis

In [0]:
from pyspark.sql.functions import round, sum

monthly_sales = (
    sales_trend
    .groupBy("Year", "Month")
    .agg(
        round(sum("Revenue"), 0).cast("long").alias("MonthlyRevenue")
    )
    .orderBy("Year", "Month")
)

monthly_sales.show(20)

In [0]:
monthly_sales.count()

In [0]:
clean_df.write.mode("overwrite").parquet(
    "/Volumes/customeranalyticsdb/default/raw-data/clean_online_retail"
)

In [0]:
df.show(5)

In [0]:
clean_df.show(5)

## 8. Conclusion

Key Findings:

- Processed 541,909 retail transactions
- Cleaned dataset to 392,692 records
- Generated £8.89M total revenue insights
- Identified top customers and top-performing countries
- Analyzed monthly sales trends using PySpark